# MultiIndex (Hierarchical Index)

Sometimes one label isn't enough. A **MultiIndex** gives rows (or columns) multiple levels — like `(Date, Country)` for the Big Mac data.

A MultiIndex is *not* a new data type; it's the same index but made of tuples. Selecting from it uses those tuples.

In [1]:
import pandas as pd

In [2]:
# Local execution already has the data files.
# On Colab (which opens just this notebook) we download them:
import os
import urllib.request

NEEDS_FILES = ['bigmac.csv', 'worldstats.csv', 'salesmen.csv', 'foods.csv', 'quarters.csv']

for _f in NEEDS_FILES:
    if not os.path.exists(_f):
        urllib.request.urlretrieve('https://raw.githubusercontent.com/sivabalanb/Data-Analysis-with-Pandas-and-Python/master/' + _f, _f)
        print('downloaded', _f)


## The Big Mac dataset

`bigmac.csv` records the price of a Big Mac in many countries over time. Natural index: `(Date, Country)`.

In [3]:
bigmac = pd.read_csv("bigmac.csv", parse_dates=["Date"])
bigmac.head()

/tmp/ipykernel_42703/1821171569.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  bigmac = pd.read_csv("bigmac.csv", parse_dates=["Date"])


,Date,Country,Price in US Dollars
0,2016-01-01,Argentina,2.39
1,2016-01-01,Australia,3.74
2,2016-01-01,Brazil,3.35
3,2016-01-01,Britain,4.22
4,2016-01-01,Canada,4.14


In [4]:
bigmac.dtypes

Date                   datetime64[us]
Country                           str
Price in US Dollars           float64
dtype: object

In [5]:
bigmac.info()

<class 'pandas.DataFrame'>
RangeIndex: 652 entries, 0 to 651
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Date                 652 non-null    datetime64[us]
 1   Country              652 non-null    str           
 2   Price in US Dollars  652 non-null    float64       
dtypes: datetime64[us](1), float64(1), str(1)
memory usage: 15.4 KB


In [6]:
bigmac["Country"].nunique()

58

In [7]:
bigmac["Country"] = bigmac["Country"].astype("category")

## Building a MultiIndex

`set_index(["Date", "Country"])` turns two columns into the index levels. `sort_index()` sorts by all levels — required for clean `loc` selection on a MultiIndex.

In [8]:
bigmac.set_index(keys = ["Date","Country"], inplace=True)

In [9]:
bigmac.sort_index(inplace=True)

In [10]:
bigmac.index.names

FrozenList(['Date', 'Country'])

In [11]:
type(bigmac.index)

pandas.MultiIndex

In [12]:
bigmac = pd.read_csv("bigmac.csv", parse_dates=["Date"], index_col=["Date","Country"])
bigmac.sort_index(inplace=True)
bigmac.head()

/tmp/ipykernel_42703/3285740269.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  bigmac = pd.read_csv("bigmac.csv", parse_dates=["Date"], index_col=["Date","Country"])


Price in US Dollars
Date       Country                       
2010-01-01 Argentina                 1.84
           Australia                 3.98
           Brazil                    4.76
           Britain                   3.67
           Canada                    3.97

## Working with levels

- `.index.names` — level names
- `get_level_values(0)` — the values of one level
- `set_names(...)` — rename levels in place

In [13]:
bigmac.index.get_level_values(0)

DatetimeIndex(['2010-01-01', '2010-01-01', '2010-01-01', '2010-01-01',
               '2010-01-01', '2010-01-01', '2010-01-01', '2010-01-01',
               '2010-01-01', '2010-01-01',
               ...
               '2016-01-01', '2016-01-01', '2016-01-01', '2016-01-01',
               '2016-01-01', '2016-01-01', '2016-01-01', '2016-01-01',
               '2016-01-01', '2016-01-01'],
              dtype='datetime64[us]', name='Date', length=652, freq=None)

In [14]:
bigmac.index.set_names(["Date","Country"], inplace=True)

In [15]:
bigmac.head()

Price in US Dollars
Date       Country                       
2010-01-01 Argentina                 1.84
           Australia                 3.98
           Brazil                    4.76
           Britain                   3.67
           Canada                    3.97

In [16]:
bigmac.sort_index(ascending=[True,False])

Price in US Dollars
Date       Country                           
2010-01-01 Uruguay                       3.32
           United States                 3.58
           Ukraine                       1.83
           UAE                           2.99
           Turkey                        3.83
...                                       ...
2016-01-01 Brazil                        3.35
           Belgium                       4.25
           Austria                       3.76
           Australia                     3.74
           Argentina                     2.39

[652 rows x 1 columns]

## Selecting from a MultiIndex

`bigmac.loc[("2010-01-01", "Norway"), "Price in US Dollars"]` — pass a **tuple** of level values, in order. Column selection works as usual with `bigmac["Price in US Dollars"]`.

In [17]:
bigmac.loc[("2010-01-01","Norway"),"Price in US Dollars"]

np.float64(7.02)

In [18]:
bigmac.transpose()

bigmac.head()

Price in US Dollars
Date       Country                       
2010-01-01 Argentina                 1.84
           Australia                 3.98
           Brazil                    4.76
           Britain                   3.67
           Canada                    3.97

In [19]:
bigmac["Price in US Dollars"]

Date        Country      
2010-01-01  Argentina        1.84
            Australia        3.98
            Brazil           4.76
            Britain          3.67
            Canada           3.97
                             ... 
2016-01-01  Ukraine          1.54
            United States    4.93
            Uruguay          3.74
            Venezuela        0.66
            Vietnam          2.67
Name: Price in US Dollars, Length: 652, dtype: float64

In [20]:
bigmac = pd.read_csv("bigmac.csv", parse_dates=["Date"], index_col=["Date","Country"])
bigmac.sort_index(inplace=True)
bigmac.head()

/tmp/ipykernel_42703/3285740269.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  bigmac = pd.read_csv("bigmac.csv", parse_dates=["Date"], index_col=["Date","Country"])


Price in US Dollars
Date       Country                       
2010-01-01 Argentina                 1.84
           Australia                 3.98
           Brazil                    4.76
           Britain                   3.67
           Canada                    3.97

## Reshaping: `stack` and `unstack`

- `stack()` pivots column labels *into* a new index level (long format).
- `unstack()` pivots an index level *into* columns (wide format).

`unstack` accepts the level to move (`unstack("year")`), or a list of levels. Repeated `unstack` flattens more levels each time.

In [21]:
bigmac.swaplevel()

,,Price in US Dollars
Country,Date,
Argentina,2010-01-01,1.84
Australia,2010-01-01,3.98
Brazil,2010-01-01,4.76
Britain,2010-01-01,3.67
Canada,2010-01-01,3.97
...,...,...
Ukraine,2016-01-01,1.54
United States,2016-01-01,4.93
Uruguay,2016-01-01,3.74


In [22]:
world = pd.read_csv("worldstats.csv", index_col=["country","year"])
world.head()

Population           GDP
country    year                           
Arab World 2015  392022276.0  2.530102e+12
           2014  384222592.0  2.873600e+12
           2013  376504253.0  2.846994e+12
           2012  368802611.0  2.773270e+12
           2011  361031820.0  2.497945e+12

In [23]:
world.stack()

country     year            
Arab World  2015  Population    3.920223e+08
                  GDP           2.530102e+12
            2014  Population    3.842226e+08
                  GDP           2.873600e+12
            2013  Population    3.765043e+08
                                    ...     
Zimbabwe    1962  GDP           1.117602e+09
            1961  Population    3.876638e+06
                  GDP           1.096647e+09
            1960  Population    3.752390e+06
                  GDP           1.052990e+09
Length: 22422, dtype: float64

In [24]:
world.info()

<class 'pandas.DataFrame'>
MultiIndex: 11211 entries, ('Arab World', np.int64(2015)) to ('Zimbabwe', np.int64(1960))
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Population  11211 non-null  float64
 1   GDP         11211 non-null  float64
dtypes: float64(2)
memory usage: 218.6+ KB


In [25]:
world.head()

Population           GDP
country    year                           
Arab World 2015  392022276.0  2.530102e+12
           2014  384222592.0  2.873600e+12
           2013  376504253.0  2.846994e+12
           2012  368802611.0  2.773270e+12
           2011  361031820.0  2.497945e+12

In [26]:
type(world.stack().to_frame())

pandas.DataFrame

In [27]:
s = world.stack()

In [28]:
s.unstack()

Population           GDP
country     year                          
Afghanistan 1960   8994793.0  5.377778e+08
            1961   9164945.0  5.488889e+08
            1962   9343772.0  5.466667e+08
            1963   9531555.0  7.511112e+08
            1964   9728645.0  8.000000e+08
...                      ...           ...
Zimbabwe    2011  14255592.0  1.095623e+10
            2012  14565482.0  1.239272e+10
            2013  14898092.0  1.349023e+10
            2014  15245855.0  1.419691e+10
            2015  15602751.0  1.389294e+10

[11211 rows x 2 columns]

In [29]:
s.unstack().unstack().unstack()

            year  country           
Population  1960  Afghanistan           8.994793e+06
                  Albania                        NaN
                  Algeria               1.112489e+07
                  Andorra                        NaN
                  Angola                         NaN
                                            ...     
GDP         2015  West Bank and Gaza    1.267740e+10
                  World                 7.343364e+13
                  Yemen, Rep.                    NaN
                  Zambia                2.120156e+10
                  Zimbabwe              1.389294e+10
Length: 28224, dtype: float64

In [30]:
s.unstack(2)
s.unstack("year")

year                            1960          1961          1962  \
country                                                            
Afghanistan Population  8.994793e+06  9.164945e+06  9.343772e+06   
            GDP         5.377778e+08  5.488889e+08  5.466667e+08   
Albania     Population           NaN           NaN           NaN   
            GDP                  NaN           NaN           NaN   
Algeria     Population  1.112489e+07  1.140486e+07  1.169015e+07   
...                              ...           ...           ...   
Yemen, Rep. GDP                  NaN           NaN           NaN   
Zambia      Population  3.049586e+06  3.142848e+06  3.240664e+06   
            GDP         6.987397e+08  6.823597e+08  6.792797e+08   
Zimbabwe    Population  3.752390e+06  3.876638e+06  4.006262e+06   
            GDP         1.052990e+09  1.096647e+09  1.117602e+09   

year                            1963          1964          1965  \
country                                                            
Afghanistan Population  9.531555e+06  9.728645e+06  9.935358e+06   
            GDP         7.511112e+08  8.000000e+08  1.006667e+09   
Albania     Population           NaN           NaN           NaN   
            GDP                  NaN           NaN           NaN   
Algeria     Population  1.198513e+07  1.229597e+07  1.262695e+07   
...                              ...           ...           ...   
Yemen, Rep. GDP                  NaN           NaN           NaN   
Zambia      Population  3.342894e+06  3.449266e+06  3.559687e+06   
            GDP         7.043397e+08  8.226397e+08  1.061200e+09   
Zimbabwe    Population  4.140804e+06  4.279561e+06  4.422132e+06   
            GDP         1.159512e+09  1.217138e+09  1.311436e+09   

year                            1966          1967          1968  \
country                                                            
Afghanistan Population  1.014884e+07  1.036860e+07  1.059979e+07   
            GDP         1.400000e+09  1.673333e+09  1.373333e+09   
Albania     Population           NaN           NaN           NaN   
            GDP                  NaN           NaN           NaN   
Algeria     Population  1.298027e+07  1.335420e+07  1.374438e+07   
...                              ...           ...           ...   
Yemen, Rep. GDP                  NaN           NaN           NaN   
Zambia      Population  3.674088e+06  3.792864e+06  3.916928e+06   
            GDP         1.239000e+09  1.340639e+09  1.573739e+09   
Zimbabwe    Population  4.568320e+06  4.718612e+06  4.874113e+06   
            GDP         1.281750e+09  1.397002e+09  1.479600e+09   

year                            1969  ...          2006          2007  \
country                               ...                               
Afghanistan Population  1.084951e+07  ...  2.518362e+07  2.587754e+07   
            GDP         1.408889e+09  ...  7.057598e+09  9.843842e+09   
Albania     Population           NaN  ...  2.992547e+06  2.970017e+06   
            GDP                  NaN  ...  8.992642e+09  1.070101e+10   
Algeria     Population  1.414444e+07  ...  3.374933e+07  3.426197e+07   
...                              ...  ...           ...           ...   
Yemen, Rep. GDP                  NaN  ...  1.908173e+10  2.563367e+10   
Zambia      Population  4.047479e+06  ...  1.238151e+07  1.273868e+07   
            GDP         1.926399e+09  ...  1.275686e+10  1.405696e+10   
Zimbabwe    Population  5.036321e+06  ...  1.312794e+07  1.329780e+07   
            GDP         1.747999e+09  ...  5.443896e+09  5.291950e+09   

year                            2008          2009          2010  \
country                                                            
Afghanistan Population  2.652874e+07  2.720729e+07  2.796221e+07   
            GDP         1.019053e+10  1.248694e+10  1.593680e+10   
Albania     Population  2.947314e+06  2.927519e+06  2.913021e+06   
            GDP         1.288135e+10  1.204421e+10  1.1

In [31]:
s.unstack(level = ["country","year"])

country       Arab World                                            \
year                2015          2014          2013          2012   
Population  3.920223e+08  3.842226e+08  3.765043e+08  3.688026e+08   
GDP         2.530102e+12  2.873600e+12  2.846994e+12  2.773270e+12   

country                                                             \
year                2011          2010          2009          2008   
Population  3.610318e+08  3.531122e+08  3.450542e+08  3.368865e+08   
GDP         2.497945e+12  2.103825e+12  1.798878e+12  2.081343e+12   

country                                 ...      Zimbabwe                \
year                2007          2006  ...          1969          1968   
Population  3.287666e+08  3.209067e+08  ...  5.036321e+06  4.874113e+06   
GDP         1.641666e+12  1.404190e+12  ...  1.747999e+09  1.479600e+09   

country                                                             \
year                1967          1966          1965          1964   
Population  4.718612e+06  4.568320e+06  4.422132e+06  4.279561e+06   
GDP         1.397002e+09  1.281750e+09  1.311436e+09  1.217138e+09   

country                                                             
year                1963          1962          1961          1960  
Population  4.140804e+06  4.006262e+06  3.876638e+06  3.752390e+06  
GDP         1.159512e+09  1.117602e+09  1.096647e+09  1.052990e+09  

[2 rows x 11211 columns]

## `fill_value` in unstack

When a combination has no data, `unstack` fills it with `NaN` — or with your chosen value: `unstack(level=["year"], fill_value=0)`.

In [32]:
t = s.unstack(level = ["year"], fill_value = 0)

In [33]:
t.head()


year                            1960          1961          1962  \
country                                                            
Afghanistan Population  8.994793e+06  9.164945e+06  9.343772e+06   
            GDP         5.377778e+08  5.488889e+08  5.466667e+08   
Albania     Population  0.000000e+00  0.000000e+00  0.000000e+00   
            GDP         0.000000e+00  0.000000e+00  0.000000e+00   
Algeria     Population  1.112489e+07  1.140486e+07  1.169015e+07   

year                            1963          1964          1965  \
country                                                            
Afghanistan Population  9.531555e+06  9.728645e+06  9.935358e+06   
            GDP         7.511112e+08  8.000000e+08  1.006667e+09   
Albania     Population  0.000000e+00  0.000000e+00  0.000000e+00   
            GDP         0.000000e+00  0.000000e+00  0.000000e+00   
Algeria     Population  1.198513e+07  1.229597e+07  1.262695e+07   

year                            1966          1967          1968  \
country                                                            
Afghanistan Population  1.014884e+07  1.036860e+07  1.059979e+07   
            GDP         1.400000e+09  1.673333e+09  1.373333e+09   
Albania     Population  0.000000e+00  0.000000e+00  0.000000e+00   
            GDP         0.000000e+00  0.000000e+00  0.000000e+00   
Algeria     Population  1.298027e+07  1.335420e+07  1.374438e+07   

year                            1969  ...          2006          2007  \
country                               ...                               
Afghanistan Population  1.084951e+07  ...  2.518362e+07  2.587754e+07   
            GDP         1.408889e+09  ...  7.057598e+09  9.843842e+09   
Albania     Population  0.000000e+00  ...  2.992547e+06  2.970017e+06   
            GDP         0.000000e+00  ...  8.992642e+09  1.070101e+10   
Algeria     Population  1.414444e+07  ...  3.374933e+07  3.426197e+07   

year                            2008          2009          2010  \
country                                                            
Afghanistan Population  2.652874e+07  2.720729e+07  2.796221e+07   
            GDP         1.019053e+10  1.248694e+10  1.593680e+10   
Albania     Population  2.947314e+06  2.927519e+06  2.913021e+06   
            GDP         1.288135e+10  1.204421e+10  1.192695e+10   
Algeria     Population  3.481106e+07  3.540179e+07  3.603616e+07   

year                            2011          2012          2013  \
country                                                            
Afghanistan Population  2.880917e+07  2.972680e+07  3.068250e+07   
            GDP         1.793024e+10  2.053654e+10  2.004633e+10   
Albania     Population  2.904780e+06  2.900247e+06  2.896652e+06   
            GDP         1.289087e+10  1.231978e+10  1.278103e+10   
Algeria     Population  3.671713e+07  3.743943e+07  3.818614e+07   

year                            2014          2015  
country                                             
Afghanistan Population  3.162751e+07  3.252656e+07  
            GDP         2.005019e+10  1.919944e+10  
Albania     Population  2.893654e+06  2.889167e+06  
            GDP         1.327796e+10  1.145560e+10  
Algeria     Population  3.893433e+07  3.966652e+07  

[5 rows x 56 columns]

## `pivot` — reshape from long to wide

`sales.pivot(index="Date", columns="Salesman", values="Revenue")` turns each Salesman into a column and each Date into a row.

**`pivot` fails if a (index, columns) pair repeats.** That's when you need `pivot_table`, which aggregates duplicates.

In [34]:
sales = pd.read_csv("salesmen.csv", parse_dates = ["Date"])
sales["Salesman"] = sales["Salesman"].astype("category")

/tmp/ipykernel_42703/857146905.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  sales = pd.read_csv("salesmen.csv", parse_dates = ["Date"])


In [35]:
sales.head()

,Date,Salesman,Revenue
0,2016-01-01,Bob,7172
1,2016-01-02,Bob,6362
2,2016-01-03,Bob,5982
3,2016-01-04,Bob,7917
4,2016-01-05,Bob,7837


In [36]:
sales.pivot(index="Date", columns="Salesman", values="Revenue")

Salesman,Bob,Dave,Jeb,Oscar,Ronald
Date,,,,,
2016-01-01,7172,1864,4430,5250,2639
2016-01-02,6362,8278,8026,8661,4951
2016-01-03,5982,4226,5188,7075,2703
2016-01-04,7917,3868,3144,2524,4258
2016-01-05,7837,2287,938,2793,7771
...,...,...,...,...,...
2016-12-27,2045,2843,6666,835,2981
2016-12-28,100,8888,1243,3073,6129
2016-12-29,4115,9490,3498,6424,7662


## `pivot_table` — pivot + aggregate

`pivot_table(values="Spend", index=[...], columns="City", aggfunc="sum")` is a pivot that *combines* duplicates (here by summing). The Swiss-army knife of reshaping.

In [37]:
foods = pd.read_csv("foods.csv")
foods.head()

,First Name,Gender,City,Frequency,Item,Spend
0,Wanda,Female,Stamford,Weekly,Burger,15.66
1,Eric,Male,Stamford,Daily,Chalupa,10.56
2,Charles,Male,New York,Never,Sushi,42.14
3,Anna,Female,Philadelphia,Once,Ice Cream,11.01
4,Deborah,Female,Philadelphia,Daily,Chalupa,23.49


In [38]:
foods.pivot_table(values="Spend", index=["Gender","Item"], columns="City", aggfunc="sum")

City              New York  Philadelphia  Stamford
Gender Item                                       
Female Burger      1239.04       1639.24   1216.02
       Burrito      978.95       1458.76   1820.11
       Chalupa      876.58       1673.33   1602.35
       Donut       1446.78       1639.26   1656.96
       Ice Cream   1521.62       1479.22   1032.03
       Sushi       1480.29       1742.88   1459.91
Male   Burger      1294.09        938.18   1439.16
       Burrito     1399.40       1312.93   1300.29
       Chalupa     1227.77       1114.23   1150.26
       Donut       1345.27       1249.36   1421.13
       Ice Cream   1603.63       2191.27   1059.22
       Sushi       1396.15       1395.88   1267.82

## `melt` — wide to long

`melt(id_vars="Salesman", var_name="Quarter", value_name="Revenue")` collapses many columns into two: a *variable* column and a *value* column. The reverse of `pivot`. Plotting libraries and SQL `UNPIVOT` expect this shape.

In [39]:
sales = pd.read_csv("quarters.csv")
sales.head()

,Salesman,Q1,Q2,Q3,Q4
0,Boris,602908,233879,354479,32704
1,Bob,43790,514863,297151,544493
2,Tommy,392668,113579,430882,247231
3,Travis,834663,266785,749238,570524
4,Donald,580935,411379,110390,651572


In [40]:
sales.melt(id_vars="Salesman",var_name="Quater", value_name="Revenue")

,Salesman,Quater,Revenue
0,Boris,Q1,602908
1,Bob,Q1,43790
2,Tommy,Q1,392668
3,Travis,Q1,834663
4,Donald,Q1,580935
5,Ted,Q1,656644
6,Jeb,Q1,486141
7,Stacy,Q1,479662
8,Morgan,Q1,992673
9,Boris,Q2,233879
